In [8]:
import pandas as pd
from mlxtend.frequent_patterns import apriori, fpgrowth, association_rules
import time

df = pd.read_csv('../data/processed/accidents_cleaned.csv')
print("Shape:", df.shape)


def time_bucket(hour):
    if 5 <= hour < 12:
        return 'Morning'
    elif 12 <= hour < 17:
        return 'Afternoon'
    elif 17 <= hour < 21:
        return 'Evening'
    else:
        return 'Night'

df['Time_Bucket'] = df['Hour'].apply(time_bucket)


temp_median = df['Temperature(F)'].median()
df['Temp_Level'] = df['Temperature(F)'].apply(lambda x: 'Low_Temp' if x < temp_median else 'High_Temp')

df['DayNight'] = df['Sunrise_Sunset'].map({0: 'Day', 1: 'Night'})
df['Severity_Level'] = df['Severity_Binary'].map({0: 'Minor_Severity', 1: 'Major_Severity'})

weather_cols = [c for c in df.columns if c.startswith('Weather_Condition_')]

def get_weather(row):
    for col in weather_cols:
        if row[col]:
            return col.replace('Weather_Condition_', '')
    return 'Clear'

df['Weather_Label'] = df[weather_cols].apply(get_weather, axis=1)


transaction_cols = ['Time_Bucket', 'Temp_Level', 'DayNight', 'Severity_Level', 'Weather_Label']
basket = pd.get_dummies(df[transaction_cols].astype(str))
basket = basket.astype(bool)

print("\nTransaction basket shape:", basket.shape)


start = time.time()
apriori_itemsets = apriori(basket, min_support=0.03, use_colnames=True)
apriori_time = time.time() - start
print(f"\nApriori found {len(apriori_itemsets)} frequent itemsets in {apriori_time:.2f} seconds")

apriori_rules = association_rules(apriori_itemsets, metric="confidence", min_threshold=0.3)
apriori_rules = apriori_rules.sort_values('lift', ascending=False)
print(f"Apriori generated {len(apriori_rules)} rules")


start = time.time()
fpgrowth_itemsets = fpgrowth(basket, min_support=0.03, use_colnames=True)
fpgrowth_time = time.time() - start
print(f"\nFP-Growth found {len(fpgrowth_itemsets)} frequent itemsets in {fpgrowth_time:.2f} seconds")

fpgrowth_rules = association_rules(fpgrowth_itemsets, metric="confidence", min_threshold=0.3)
fpgrowth_rules = fpgrowth_rules.sort_values('lift', ascending=False)
print(f"FP-Growth generated {len(fpgrowth_rules)} rules")


print("\n--- Speed Comparison ---")
print(f"Apriori:   {apriori_time:.3f} seconds")
print(f"FP-Growth: {fpgrowth_time:.3f} seconds")


def show_severity_rules(rules, name):
    severity_rules = rules[rules['consequents'].astype(str).str.contains('Major_Severity')]
    severity_rules = severity_rules.sort_values('lift', ascending=False)
    print(f"\nTop {name} rules predicting Major Severity (by lift):")
    print(severity_rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(10).to_string())
    return severity_rules

apriori_severity_rules = show_severity_rules(apriori_rules, "Apriori")
fpgrowth_severity_rules = show_severity_rules(fpgrowth_rules, "FP-Growth")


apriori_rules.to_csv('../data/processed/apriori_rules.csv', index=False)
fpgrowth_rules.to_csv('../data/processed/fpgrowth_rules.csv', index=False)
print("\nSaved apriori_rules.csv and fpgrowth_rules.csv")

Shape: (155744, 19)

Transaction basket shape: (155744, 21)

Apriori found 241 frequent itemsets in 0.13 seconds
Apriori generated 807 rules

FP-Growth found 241 frequent itemsets in 0.80 seconds
FP-Growth generated 807 rules

--- Speed Comparison ---
Apriori:   0.132 seconds
FP-Growth: 0.802 seconds

Top Apriori rules predicting Major Severity (by lift):
                                                 antecedents                                                                      consequents   support  confidence      lift
620                           frozenset({Time_Bucket_Night})  frozenset({Severity_Level_Major_Severity, Temp_Level_Low_Temp, DayNight_Night})  0.048381    0.369943  3.627552
265                           frozenset({Time_Bucket_Night})                       frozenset({Severity_Level_Major_Severity, DayNight_Night})  0.061357    0.469167  3.441341
618      frozenset({Time_Bucket_Night, Temp_Level_Low_Temp})                       frozenset({Severity_Level_Major_Sever